# Chatbot Project

### Importing libraries

In [1]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
import numpy as np
nltk.download('punkt')
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.porter import PorterStemmer
!pip install TextBlob
from textblob import TextBlob
!pip install pyspellchecker
!pip install --upgrade spacy
from spellchecker import SpellChecker
import spacy
from spacy import displacy
from collections import Counter
!python -m spacy download en_core_web_sm
import en_core_web_sm
nlp = en_core_web_sm.load()
from nltk.chunk import conlltags2tree, tree2conlltags
from pprint import pprint
import random
from spacy.util import minibatch, compounding
from pathlib import Path
!pip install spacytextblob
from spacytextblob.spacytextblob import SpacyTextBlob

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\transorg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\transorg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


  Attempting uninstall: wasabi
    Found existing installation: wasabi 0.8.2
    Uninstalling wasabi-0.8.2:
      Successfully uninstalled wasabi-0.8.2
  Attempting uninstall: srsly
    Found existing installation: srsly 2.4.1
    Uninstalling srsly-2.4.1:
      Successfully uninstalled srsly-2.4.1
  Attempting uninstall: thinc
    Found existing installation: thinc 8.0.12
    Uninstalling thinc-8.0.12:
      Successfully uninstalled thinc-8.0.12
  Attempting uninstall: spacy-legacy
    Found existing installation: spacy-legacy 3.0.8
    Uninstalling spacy-legacy-3.0.8:
      Successfully uninstalled spacy-legacy-3.0.8
  Attempting uninstall: spacy
    Found existing installation: spacy 3.2.1
    Uninstalling spacy-3.2.1:
      Successfully uninstalled spacy-3.2.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
en-core-web-sm 3.2.0 requires spacy<3.3.0,>=3.2.0, but you have spacy 3.3.0 which is incompatible.
en-core-web-md 3.1.0 requires spacy<3.2.0,>=3.1.0, but you have spacy 3.3.0 which is incompatible.
en-core-web-lg 3.1.0 requires spacy<3.2.0,>=3.1.0, but you have spacy 3.3.0 which is incompatible.


  Attempting uninstall: en-core-web-sm
    Found existing installation: en-core-web-sm 3.2.0
    Uninstalling en-core-web-sm-3.2.0:
      Successfully uninstalled en-core-web-sm-3.2.0
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


2022-05-18 15:52:15.092369: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'cudart64_110.dll'; dlerror: cudart64_110.dll not found
2022-05-18 15:52:15.092404: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


### Importing training data

- An excel database has been created to store training data.
- Various user intents and associated patterns (utterances) constitutes the training data.

In [2]:
df = pd.read_excel('Intents-examples.xlsx')

### Importing customer database

- User/Customer-specific database has been created to provide customer-specific responses.
- The database comprise data fields containing basic customer information like Customer ID, Bill due, Last meter reading, etc. 

In [3]:
# importing user data file
user_data = pd.read_excel('user_data.xlsx')

### Data preprocessing

The training data comprising various intents and query examples will be used to train our Machine Learning (ML) model. To make it understandable by ML model, we first did some preprocessing steps and performed vectorization.

In [4]:
df = df.drop(df.columns[0], axis=1)
df['Intents'] = df['Intents'].fillna(method='ffill')

Here, from our training examples, we are removing numerals, converting text to lowercase, removing stopwords (except 'not'and 'you'), if any. 

In [5]:
vocab = []
corpus = []
for i in range(0, len(df)):
  review = re.sub('[^a-zA-Z]', ' ', df['Examples'][i])
  review = review.lower()
  review = review.split()
  
  for word in review:
    vocab.append(word)
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  all_stopwords.remove('you')
  review = [(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

In [6]:
corpus

['hi',
 'hello',
 'you',
 'hey',
 'hi',
 'whats',
 'hey',
 'hii',
 'hiii',
 'good morning',
 'good evening',
 'greetings',
 'hey',
 'last month bill amount',
 'payable amount',
 'payment due',
 'due amount',
 'total amount due',
 'total payment due',
 'last month invoice',
 'total last month bill',
 'much need pay last month',
 'much bill last month',
 'payable amount',
 'pending bill amount',
 'payment due',
 'due amount',
 'due payable amount',
 'due payment',
 'amount payable',
 'monthly bill',
 'monthly growth',
 'mom growth',
 'much mom growth',
 'much usage increased month',
 'much month month growth',
 'monthly growth month month',
 'show monthly growth',
 'display month month growth',
 'tell month month growth',
 'increase monthly usage',
 'mom growth',
 'mom usage',
 'consumption pattern last months',
 'months consumption pattern',
 'year consumption pattern',
 'utilization last months',
 'show consumption pattern last months',
 'usage months',
 'usage year',
 'consumption tre

In [6]:
len(corpus)

160

Creating a Vocabulary

In [7]:
vocab = list(dict.fromkeys(vocab))
vocab = pd.DataFrame(vocab)
vocab.to_excel('vocab.xlsx', index=False, header=False)

In [8]:
vocab

,0
0,hi
1,hello
2,how
3,are
4,you
...,...
161,cnsmptin
162,pattrn
163,mnthly
164,pymnt


### Vectorization

Transforming each example into a vector.

In [8]:
vectorize = TfidfVectorizer()
vectors = vectorize.fit_transform(corpus)
x = vectors.toarray()

In [12]:
x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
x.shape

(160, 124)

Transforming the intents of all examples into vectors.

In [9]:
from sklearn.preprocessing import LabelBinarizer
enc = LabelBinarizer()
y = df['Intents']
y = enc.fit_transform(y)

In [13]:
y

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [15]:
y.shape

(160, 12)

Splitting the data into training and testing sets.

In [10]:
from sklearn.model_selection import train_test_split
# splitting data into 80% train and 20% test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.1, random_state = 0, shuffle=True) 

### Training Neural Network (NN) model for Intent Classification

Importing required libraries and methods

In [11]:
import tensorflow as tf 
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense, Dropout

Creating Early Stopping and Model Checkpoint callbacks. 

In [12]:
from keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler, Callback
from keras.models import load_model
import keras.backend as K

Configuring above callbacks and creating Sequential constructor and passing a list of layers and setting parameters. At the end we are fitting the model by passing x_train and y_train data and checking its performance over test dataset.

In [13]:
epochs = 200

mc = ModelCheckpoint('_best_model.h5', monitor='val_loss', mode='min', verbose=1, save_best_only=True)

def step_decay_schedule(initial_lr=1e-3, decay_factor=0.75, step_size=10):
    def schedule(epoch):
        return initial_lr * (decay_factor ** np.floor(epoch/step_size))   
    return LearningRateScheduler(schedule)

class LearningRateMonitor(Callback):
  # start of training
  def on_train_begin(self, logs={}):
    self.lrates = list()

  # end of each training epoch
  def on_epoch_end(self, epoch, logs={}):
    # get and store the learning rate
    optimizer = self.model.optimizer
    lrate = float(K.get_value(self.model.optimizer.lr))
    self.lrates.append(lrate)

def run_model(model, default_lr=0.001, patience=50, baseline=0.7,min_delta=0,decay=False,decay_factor=0.95,step_size=1):
  K.set_value(model.optimizer.learning_rate, default_lr, )
  es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=patience, baseline=baseline, min_delta=min_delta)
  lrm = LearningRateMonitor()
  if decay:
    lr_sched = step_decay_schedule(initial_lr=default_lr, decay_factor=decay_factor, step_size=step_size)
    history = model.fit(x_train, y_train, epochs=200, validation_data=(x_test, y_test), verbose=2, shuffle=False, callbacks=[es,mc,lrm,lr_sched])
  else:
    history = model.fit(x_train, y_train, epochs=200, validation_data=(x_test, y_test), verbose=2, shuffle=False, callbacks=[es,mc,lrm])


# the deep learning model
model = Sequential()
model.add(Dense(128, input_shape=(len(x_train[0]),), activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(len(y_train[0]), activation = "softmax"))
adam = tf.keras.optimizers.Adam(learning_rate=0.01, decay=1e-6)
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=["accuracy"])
#print(model.summary())
# history = model.fit(x=x_train, y=y_train, validation_data=(x_test, y_test), epochs=epochs, verbose=1)

run_model(model)
run_model(model, decay=True)
run_model(model, default_lr=3e-4)
model = load_model('_best_model.h5')

Epoch 1/200
5/5 - 1s - loss: 2.4794 - accuracy: 0.0833 - val_loss: 2.4649 - val_accuracy: 0.1250

Epoch 00001: val_loss improved from inf to 2.46489, saving model to _best_model.h5
Epoch 2/200
5/5 - 0s - loss: 2.4574 - accuracy: 0.1944 - val_loss: 2.4497 - val_accuracy: 0.1250

Epoch 00002: val_loss improved from 2.46489 to 2.44965, saving model to _best_model.h5
Epoch 3/200
5/5 - 0s - loss: 2.4403 - accuracy: 0.1736 - val_loss: 2.4322 - val_accuracy: 0.1875

Epoch 00003: val_loss improved from 2.44965 to 2.43222, saving model to _best_model.h5
Epoch 4/200
5/5 - 0s - loss: 2.4137 - accuracy: 0.1944 - val_loss: 2.4140 - val_accuracy: 0.2500

Epoch 00004: val_loss improved from 2.43222 to 2.41399, saving model to _best_model.h5
Epoch 5/200
5/5 - 0s - loss: 2.3836 - accuracy: 0.2292 - val_loss: 2.3954 - val_accuracy: 0.2500

Epoch 00005: val_loss improved from 2.41399 to 2.39537, saving model to _best_model.h5
Epoch 6/200
5/5 - 0s - loss: 2.3616 - accuracy: 0.1944 - val_loss: 2.3749 - val


Epoch 00045: val_loss improved from 0.54976 to 0.52465, saving model to _best_model.h5
Epoch 46/200
5/5 - 0s - loss: 0.2795 - accuracy: 0.9722 - val_loss: 0.4987 - val_accuracy: 0.8750

Epoch 00046: val_loss improved from 0.52465 to 0.49873, saving model to _best_model.h5
Epoch 47/200
5/5 - 0s - loss: 0.3234 - accuracy: 0.9792 - val_loss: 0.4770 - val_accuracy: 0.8750

Epoch 00047: val_loss improved from 0.49873 to 0.47704, saving model to _best_model.h5
Epoch 48/200
5/5 - 0s - loss: 0.2975 - accuracy: 0.9722 - val_loss: 0.4607 - val_accuracy: 0.9375

Epoch 00048: val_loss improved from 0.47704 to 0.46072, saving model to _best_model.h5
Epoch 49/200
5/5 - 0s - loss: 0.2414 - accuracy: 0.9653 - val_loss: 0.4446 - val_accuracy: 0.8750

Epoch 00049: val_loss improved from 0.46072 to 0.44461, saving model to _best_model.h5
Epoch 50/200
5/5 - 0s - loss: 0.2714 - accuracy: 0.9722 - val_loss: 0.4280 - val_accuracy: 0.9375

Epoch 00050: val_loss improved from 0.44461 to 0.42798, saving model 

5/5 - 0s - loss: 0.0592 - accuracy: 0.9931 - val_loss: 0.1925 - val_accuracy: 1.0000

Epoch 00091: val_loss improved from 0.19753 to 0.19253, saving model to _best_model.h5
Epoch 92/200
5/5 - 0s - loss: 0.0675 - accuracy: 1.0000 - val_loss: 0.1939 - val_accuracy: 1.0000

Epoch 00092: val_loss did not improve from 0.19253
Epoch 93/200
5/5 - 0s - loss: 0.0772 - accuracy: 0.9931 - val_loss: 0.1939 - val_accuracy: 1.0000

Epoch 00093: val_loss did not improve from 0.19253
Epoch 94/200
5/5 - 0s - loss: 0.0566 - accuracy: 0.9931 - val_loss: 0.1952 - val_accuracy: 1.0000

Epoch 00094: val_loss did not improve from 0.19253
Epoch 95/200
5/5 - 0s - loss: 0.0445 - accuracy: 1.0000 - val_loss: 0.1902 - val_accuracy: 1.0000

Epoch 00095: val_loss improved from 0.19253 to 0.19015, saving model to _best_model.h5
Epoch 96/200
5/5 - 0s - loss: 0.0504 - accuracy: 1.0000 - val_loss: 0.1864 - val_accuracy: 1.0000

Epoch 00096: val_loss improved from 0.19015 to 0.18643, saving model to _best_model.h5
Epoch


Epoch 00142: val_loss did not improve from 0.15356
Epoch 143/200
5/5 - 0s - loss: 0.0264 - accuracy: 0.9931 - val_loss: 0.1897 - val_accuracy: 0.8750

Epoch 00143: val_loss did not improve from 0.15356
Epoch 144/200
5/5 - 0s - loss: 0.0245 - accuracy: 1.0000 - val_loss: 0.1856 - val_accuracy: 0.8750

Epoch 00144: val_loss did not improve from 0.15356
Epoch 145/200
5/5 - 0s - loss: 0.0245 - accuracy: 1.0000 - val_loss: 0.1810 - val_accuracy: 0.8750

Epoch 00145: val_loss did not improve from 0.15356
Epoch 146/200
5/5 - 0s - loss: 0.0245 - accuracy: 0.9931 - val_loss: 0.1753 - val_accuracy: 0.9375

Epoch 00146: val_loss did not improve from 0.15356
Epoch 147/200
5/5 - 0s - loss: 0.0206 - accuracy: 1.0000 - val_loss: 0.1733 - val_accuracy: 0.9375

Epoch 00147: val_loss did not improve from 0.15356
Epoch 148/200
5/5 - 0s - loss: 0.0235 - accuracy: 1.0000 - val_loss: 0.1763 - val_accuracy: 0.9375

Epoch 00148: val_loss did not improve from 0.15356
Epoch 149/200
5/5 - 0s - loss: 0.0206 - ac


Epoch 00196: val_loss did not improve from 0.12618
Epoch 197/200
5/5 - 0s - loss: 0.0106 - accuracy: 1.0000 - val_loss: 0.1503 - val_accuracy: 0.9375

Epoch 00197: val_loss did not improve from 0.12618
Epoch 198/200
5/5 - 0s - loss: 0.0266 - accuracy: 0.9931 - val_loss: 0.1495 - val_accuracy: 0.9375

Epoch 00198: val_loss did not improve from 0.12618
Epoch 199/200
5/5 - 0s - loss: 0.0096 - accuracy: 1.0000 - val_loss: 0.1495 - val_accuracy: 0.9375

Epoch 00199: val_loss did not improve from 0.12618
Epoch 200/200
5/5 - 0s - loss: 0.0186 - accuracy: 1.0000 - val_loss: 0.1455 - val_accuracy: 0.9375

Epoch 00200: val_loss did not improve from 0.12618
Epoch 1/200
5/5 - 0s - loss: 0.0124 - accuracy: 1.0000 - val_loss: 0.1432 - val_accuracy: 0.9375

Epoch 00001: val_loss did not improve from 0.12618
Epoch 2/200
5/5 - 0s - loss: 0.0071 - accuracy: 1.0000 - val_loss: 0.1413 - val_accuracy: 0.9375

Epoch 00002: val_loss did not improve from 0.12618
Epoch 3/200
5/5 - 0s - loss: 0.0123 - accuracy


Epoch 00051: val_loss did not improve from 0.12618
Epoch 52/200
5/5 - 0s - loss: 0.0123 - accuracy: 1.0000 - val_loss: 0.1513 - val_accuracy: 0.9375

Epoch 00052: val_loss did not improve from 0.12618
Epoch 53/200
5/5 - 0s - loss: 0.0125 - accuracy: 1.0000 - val_loss: 0.1512 - val_accuracy: 0.9375

Epoch 00053: val_loss did not improve from 0.12618
Epoch 54/200
5/5 - 0s - loss: 0.0133 - accuracy: 1.0000 - val_loss: 0.1511 - val_accuracy: 0.9375

Epoch 00054: val_loss did not improve from 0.12618
Epoch 55/200
5/5 - 0s - loss: 0.0069 - accuracy: 1.0000 - val_loss: 0.1511 - val_accuracy: 0.9375

Epoch 00055: val_loss did not improve from 0.12618
Epoch 56/200
5/5 - 0s - loss: 0.0111 - accuracy: 1.0000 - val_loss: 0.1510 - val_accuracy: 0.9375

Epoch 00056: val_loss did not improve from 0.12618
Epoch 57/200
5/5 - 0s - loss: 0.0156 - accuracy: 1.0000 - val_loss: 0.1508 - val_accuracy: 0.9375

Epoch 00057: val_loss did not improve from 0.12618
Epoch 58/200
5/5 - 0s - loss: 0.0150 - accuracy:


Epoch 00041: val_loss did not improve from 0.12618
Epoch 42/200
5/5 - 0s - loss: 0.0103 - accuracy: 1.0000 - val_loss: 0.1586 - val_accuracy: 0.9375

Epoch 00042: val_loss did not improve from 0.12618
Epoch 43/200
5/5 - 0s - loss: 0.0092 - accuracy: 1.0000 - val_loss: 0.1588 - val_accuracy: 0.9375

Epoch 00043: val_loss did not improve from 0.12618
Epoch 44/200
5/5 - 0s - loss: 0.0136 - accuracy: 1.0000 - val_loss: 0.1583 - val_accuracy: 0.9375

Epoch 00044: val_loss did not improve from 0.12618
Epoch 45/200
5/5 - 0s - loss: 0.0112 - accuracy: 1.0000 - val_loss: 0.1567 - val_accuracy: 0.9375

Epoch 00045: val_loss did not improve from 0.12618
Epoch 46/200
5/5 - 0s - loss: 0.0081 - accuracy: 1.0000 - val_loss: 0.1547 - val_accuracy: 0.9375

Epoch 00046: val_loss did not improve from 0.12618
Epoch 47/200
5/5 - 0s - loss: 0.0046 - accuracy: 1.0000 - val_loss: 0.1540 - val_accuracy: 0.9375

Epoch 00047: val_loss did not improve from 0.12618
Epoch 48/200
5/5 - 0s - loss: 0.0158 - accuracy:

Model performance

In [14]:
y_pred = model.predict(x_test)
y_pred = enc.inverse_transform(y_pred)
y_test = enc.inverse_transform(y_test)

import sklearn.metrics as metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
accuracy = metrics.accuracy_score(y_test, y_pred)
accuracy

1.0

## Correct Spelling Function

In [15]:
import re
from collections import Counter

def words(text): 
  return re.findall(r'\w+', text.lower())

WORDS = Counter(vocab[0])

def P(word, N=sum(WORDS.values())): 
    "Probability of `word`."
    return WORDS[word] / N

def correction(word): 
    "Most probable spelling correction for word."
    return max(candidates(word), key=P)

def candidates(word): 
    "Generate possible spelling corrections for word."
    return (known([word]) or known(edits1(word)) or known(edits2(word)) or [word])

def known(words): 
    "The subset of `words` that appear in the dictionary of WORDS."
    return set(w for w in words if w in WORDS)

def edits1(word):
    "All edits that are one edit away from `word`."
    letters    = 'abcdefghijklmnopqrstuvwxyz'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(word): 
    "All edits that are two edits away from `word`."
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))

## Functions defined for chatbot

In [16]:
def predict(text):
  text = re.sub('[^a-zA-Z]', ' ', text)
  text = text.lower()
  text = text.split()
  text = [(word) for word in text if not word in set(all_stopwords)]
  misspelled = SpellChecker().unknown(text)
  for word in misspelled:
    text = [correction(word) if x==word else x for x in text]
  text = ' '.join(text)
  text = vectorize.transform([text])
  text = text.toarray()
  probs = model.predict(text).reshape(df['Intents'].nunique(),)
  dict(zip(enc.classes_, probs))
  probs = sorted(dict(zip(enc.classes_, probs)).items(), key=lambda x:x[1], reverse=True)
  if probs[0][1]<0.5:
    return 'fallback'
  else: return probs[0][0]

In [47]:
def predict_proba(message):
  text = re.sub('[^a-zA-Z]', ' ', message)
  text = text.lower()
  text = text.split()
  t = [(word) for word in text if not word in set(all_stopwords)]

  misspelled = SpellChecker().unknown(t)
  #print('misspelled words: ' + str(misspelled))
  for word in misspelled:
    t = [correction(word) if x==word else x for x in t]
  #print(t)
  t = ' '.join(t)
  text = vectorize.transform([t])
  text = text.toarray()
  probs = model.predict(text).reshape(df['Intents'].nunique(),)
  dict(zip(enc.classes_, probs))
  probs = sorted(dict(zip(enc.classes_, probs)).items(), key=lambda x:x[1], reverse=True)
  #and misspelled is none

  if probs[0][1]>0.90 and len(misspelled)==0 and not (nlp(message).ents):
    if message not in df['Examples'].unique():
      df.loc[len(df.index)] = [probs[0][0], message] 
      df.to_excel('Intents-examples.xlsx')
      #print('added ' + str(message))
  elif probs[0][1]<0.40:
    return 'fallback'
  return probs

In [18]:
def greet():
  if memory['name'] is None:
    print("Hi, I am an interactive Chatbot. \nI am here to help you with any questions you may have about" + " utlity, account creation, payment & refund "+ "and" + " complaint registration!"+"\nWhat can I help you with today?")
  else:
    txt1 = "Hi "+ str(memory['name']) + ", I am an interactive Chatbot. \nI am here to help you with any questions you may have about" + " utlity, account creation, payment & refund "+ "and" + " complaint registration!"+"\nWhat can I help you with today?"
    print(txt1)

### Input validation functions

#### Customer ID validation

Rule: A Customer ID should be of 5 digits. No alphabets should be present in an ID.

Output:

- If length of an entered ID is less than 5, it displays one warning message regarding the incorrect length.
- If ID doesn't exist in our Customer Database then it displays a different warning message to register or try entering again.

In [19]:
def validate_id(id):
    p = re.compile('^[0-9]*$')
    if (len(id) != 5):
        print('Length of ID must be 5')
        return False
    if (int(id) not in user_data['id'].unique()):
        print('ID doesn\'t exist. Please register or try again ')
        return False
    if(id == None):
        return False
    if(re.search(p, id)):
        return True
    else:
        return False

#### PAN validation

Rule: A PAN no. is an alphanumeric code of 10 characters. First 5 characters should be a combination of alphabets. Next 4 are a combination of digits. Last character should be an alphabet. 

Output: If length of an entered code is not equal to 10, it displays a warning message regarding the incorrect length and asks the user response again.

In [20]:
def validate_pan(panCardNo):
    p = re.compile("[A-Z]{5}[0-9]{4}[A-Z]{1}")
    if(panCardNo == None):
        return False
    if(len(panCardNo) != 10):
        print('Length of Pan card must be 10')
        return False
    if(re.search(p, panCardNo)):
        return True
    else:
        return False

#### Mobile No. validation

Rule: A Mobile no. should be of 10 digits 

Output: If length of an entered no. is not equal to 10, it displays a warning message regarding the incorrect length and asks the user response again.

In [21]:
def validate_mobile(m):
    if(len(m) != 10):
        print('Length of mobile number must be 10')
        return False
    Pattern = re.compile("(0|91)?[7-9][0-9]{9}")
    return Pattern.match(m)

#### PIN Code validation

Rule: A PIN Code should be of 6 digits.

Output: If length of an entered no. is not equal to 6, it displays a warning message regarding the incorrect length and asks the user response again.

In [22]:
def validate_pincode(pinCode):
    if(len(pinCode) != 6):
        print('Length of pincode must be 6')
        return False
    regex = "^[1-9]{1}[0-9]{2}\\s{0,1}[0-9]{3}$";
    p = re.compile(regex);
    if (pinCode == ''):
        return False;
    if re.match(p, pinCode) is None:
        return False
    else:
        return True

#### Aadhaar Card No. validation

Rule: An Aadhaar Card no. should be of 12 digits.

Output: If length of an entered no. is not equal to 12, it displays a warning message regarding the incorrect length and asks the user response again.

In [23]:
def validate_aadhaar(adh):
    if(len(adh) != 12):
        print('Length of aadhaar number is wrong, Please include spaces and try again')
    regex = ("^[2-9]{1}[0-9]{3}" + "[0-9]{4}[0-9]{4}$")
    p = re.compile(regex)
    if (adh == None):
        return False
    if(re.search(p, adh)):
        return True
    else:
        return False

### Entity Extraction

In [24]:
ner=nlp.get_pipe("ner")

In [25]:
TRAIN_DATA = [
              ("My name is aryan", {"entities": [(11, 16, "PERSON")]}),
              ("Name is aryan", {"entities": [(8, 13, "PERSON")]}),
              ("My name’s Aaronmi", {"entities": [(10,17, "PERSON")]}),
              ("I am aaron", {"entities": [(5,10, "PERSON")]}),
              ("my name is aryanm", {"entities": [(11, 17, "PERSON")]}),
              ("rounak is my name", {"entities": [(0, 6, "PERSON")]})
]

In [26]:
TRAIN_DATA = [
              ("My name is aryan ringshia", {"entities": [(11, 25, "PERSON")]}),
              ("Name is aryan", {"entities": [(8, 13, "PERSON")]}),
              ("My name’s Aaronmi Gupta", {"entities": [(10,23, "PERSON")]}),
              ("I am aaron", {"entities": [(5,10, "PERSON")]}),
              ("my name is aryanm", {"entities": [(11, 17, "PERSON")]}),
              ("rounak is my name", {"entities": [(0, 6, "PERSON")]}),
              ("my id is 10002", {"entities": [(9, 13, "ID")]}),
              ("my customer id is 34789", {"entities": [(18, 22, "ID")]})
]

for _, annotations in TRAIN_DATA:
  for ent in annotations.get("entities"):
    ner.add_label(ent[2])

# Disable pipeline components you dont need to change
pipe_exceptions = ["ner", "trf_wordpiecer", "trf_tok2vec"]
unaffected_pipes = [pipe for pipe in nlp.pipe_names if pipe not in pipe_exceptions]

In [27]:
from spacy.training.example import Example
with nlp.disable_pipes(*unaffected_pipes):

  # Training for 30 iterations
  for iteration in range(30):

    # shuffling examples  before every iteration
    random.shuffle(TRAIN_DATA)
    losses = {}
    # batch up the examples using spaCy's minibatch
    batches = minibatch(TRAIN_DATA, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
        for text, annotations in batch:
          example = Example.from_dict(nlp.make_doc(text), annotations)
          nlp.update([example],
                    drop=0.5,  # dropout - make it harder to memorise data
                    losses=losses,
                )
          print("Losses", losses)

Losses {'ner': 2.006861336764267}
Losses {'ner': 4.048413768646007}
Losses {'ner': 6.374790400199642}
Losses {'ner': 8.806811697378592}
Losses {'ner': 8.80881462364464}
Losses {'ner': 8.815945039485893}


C:\Users\transorg\anaconda3\lib\site-packages\spacy\training\iob_utils.py:141: UserWarning: [W030] Some entities could not be aligned in the text "my id is 10002" with entities "[(9, 13, 'ID')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\transorg\anaconda3\lib\site-packages\spacy\training\iob_utils.py:141: UserWarning: [W030] Some entities could not be aligned in the text "my customer id is 34789" with entities "[(18, 22, 'ID')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Losses {'ner': 10.106072422492513}
Losses {'ner': 10.601373812247806}
Losses {'ner': 0.08219300314080868}
Losses {'ner': 1.5147173963170033}
Losses {'ner': 1.544631677819254}
Losses {'ner': 1.5451267419429513}
Losses {'ner': 5.1653013890735}
Losses {'ner': 7.151516665576134}
Losses {'ner': 7.203863051503328}
Losses {'ner': 8.898560363901476}
Losses {'ner': 0.04654234017536396}
Losses {'ner': 2.0352062673583218}
Losses {'ner': 2.432937740211839}
Losses {'ner': 5.625041428748586}
Losses {'ner': 5.625132471193991}
Losses {'ner': 8.576174868987449}
Losses {'ner': 10.510935364198316}
Losses {'ner': 12.365108634319856}
Losses {'ner': 0.0017373201320651788}
Losses {'ner': 1.7606569609255303}
Losses {'ner': 1.760659138252521}
Losses {'ner': 2.6750991852071655}
Losses {'ner': 2.6751028341099055}
Losses {'ner': 2.6785163443497213}
Losses {'ner': 4.242747816675189}
Losses {'ner': 5.828212662532935}
Losses {'ner': 0.38206202536821365}
Losses {'ner': 0.4068124274671763}
Losses {'ner': 0.45124803281

Losses {'ner': 0.0215406931694289}
Losses {'ner': 0.02154069349787291}
Losses {'ner': 0.021540693503414473}


In [53]:
def common_entity_extraction(doc):
  doc = nlp(doc)
  #if doc.ents:
  #  displacy.render(doc, style = "ent", jupyter=True)
  
  for X in doc.ents:
    #name
    if X.label_=='PERSON':
      #memory['name'] = X.text
      memory['name'] = X.text

    if X.label_=='CARDINAL' or X.label_=='MONEY' or X.label_=='DATE':
      #check if id
      if len(X.text)==5 and str(X.text).isnumeric():
        #validate id else discard
        p = re.compile('^[0-9]*$')
        if (int(str(X.text)) in user_data['id'].unique()):
          if(re.search(p, (str(X.text)))):
            memory['id'] = int(str(X.text))

      #check if mobile
      if len(X.text)==10 and str(X.text).isnumeric():
        #validate mobile else discard
        Pattern = re.compile("(0|91)?[7-9][0-9]{9}")
        if Pattern.match(str(X.text)):
          memory['mobile'] = str(X.text)

      #check pin code
      if len(X.text)==6 and str(X.text).isnumeric():
        #validate pin code else discard
        regex = "^[1-9]{1}[0-9]{2}\\s{0,1}[0-9]{3}$";
        p = re.compile(regex);
        if re.match(p, str(X.text)) is not None:
          memory['pincode'] = int(str(X.text))


### Query response functions

#### Clearing user data stored in memory dictionary

In [29]:
def clear_memory():
    for k,v in memory.items():
        memory[k]=None

#### Fetching ID no. if already provided by user, otherwise validating using the Customer database.

In [30]:
def get_id():
    if memory['id'] is None:
        while True:
            id = input("Please enter your ID: (To go back type back) ")
            if id=='back':
              return 
            #capture digits from input and then validate
            id = [int(s) for s in id.split() if s.isdigit()]
            if id:
              id = id[0]
              if not validate_id(str(id)):
                print("Sorry, your response was not valid.")
                continue
              else:
                memory['id'] = int(id)
                break
            else:
              print('Invalid response, ID must be numeric')

#### To output the last month dues of a customer from customer database if user has entered a valid Customer ID.

In [31]:
def bill_due():
    get_id()
    if memory['id']:
      print('Checking your balance due')
      print('Balance due for ID: ' + str(memory['id']) + ' is: Rs.' + str(int(user_data[user_data['id'] == (memory['id'])].iloc[0,4])))

#### To output the last meter reading in relation to the valid Customer ID provided by a user.

In [32]:
def reading():
    get_id()
    if memory['id']:    
      print('Checking your meter reading')
      print('Meter reading for ID: ' + str(memory['id']) + ' is: ' + str(int(user_data[user_data['id'] == (memory['id'])].iloc[0,5])))

#### To output the recent month on month (MoM) usage growth in relation to the valid Customer ID provided by a user.

In [33]:
def mom():
    get_id()
    if memory['id']:
      print('Checking your Month on Month growth')
      print('Month on Month growth for ID: ' + str(memory['id']) + ' is: ' + str(int(user_data[user_data['id'] == (memory['id'])].iloc[0,7]))+"%")

#### To output the consumption trend over last 1 year for a valid Customer ID provided by a user.

In [34]:
def consumption():
    get_id()
    if memory['id']:
      print('Checking your consumption pattern over last 12 months')
      print('Consumption pattern for ID: ' + str(memory['id']) + ' is: ' + str(int(user_data[user_data['id'] == (memory['id'])].iloc[0,6]))+"%")

#### To ask and store customer information (like name, aadhaar card no., PAN, etc.) if one wants to get a new electricity connection and provide a application-completion response.

In [54]:
def new_connection():
    global user_data
    #Ask for Name , Mobile , Pin Code , Aadhaar , Pan
    if memory['name'] is None:
      while True:
        name = input("Please enter your name: (To go back type quit) ")
        if name=='quit':
            return
        name = nlp(name)
        #if name.ents:
          #pprint([(X.text, X.label_) for X in name.ents])
          #displacy.render(name, style = "ent", jupyter=True)
        for X in name.ents:
          if X.label_=='PERSON':
            memory['name'] = X.text
            break
        else:
          print('No name detected')
          continue
        if memory['name'] is not None:
          break
    
    if memory['mobile'] is None:
        while True:
            mobile = input("Please enter your mobile number: (To go back type quit) ")
            if mobile=='quit':
                for k,v in memory.items():
                    memory[k]=None
                return 
            mobile = [int(s) for s in mobile.split() if s.isdigit()]
            if mobile:
              mobile = mobile[0]
              if not validate_mobile(str(mobile)):
                print("Sorry, your response was not valid.")
                continue
              else:
                memory['mobile'] = str(mobile)
                break  
            else:
              print('Invalid response, mobile must be numeric')

    if memory['pincode'] is None:
        while True:
            pincode = input("Please enter your pincode: (To go back type quit) ")
            if pincode=='quit':
                for k,v in memory.items():
                    memory[k]=None
                return 

            pincode = [int(s) for s in pincode.split() if s.isdigit()]
            if pincode:
              pincode = pincode[0]
              if not validate_pincode(str(pincode)):
                print("Sorry, your response was not valid.")
                continue
              else:
                memory['pincode'] = str(pincode)
                break  
            else:
              print('Invalid response, pincode must be numeric')

    if memory['aadhaar'] is None:
        while True:
            aadhaar = input("Please enter your aadhaar number: (To go back type quit) ")
            if aadhaar=='quit':
                for k,v in memory.items():
                    memory[k]=None
                return 

            aadhaar = [str(s) for s in aadhaar.split() if s.isdigit()]
            aadhaar = ''.join(aadhaar)
            if aadhaar:
              if not validate_aadhaar(aadhaar):
                print("Sorry, your response was not valid.")
                continue
              else:
                memory['aadhaar'] = aadhaar
                break  
            else:
              print('Invalid response, pincode must be numeric')

    if memory['pan'] is None:
        while True:
            if memory['pan'] is not None:
              break
            pan = input("Please enter your pan number: (To go back type quit) ")
            if pan=='quit':
                for k,v in memory.items():
                    memory[k]=None
                return 
            
            pan = pan.split()
            pan = [x for x in pan if len(x)==10]
            if pan:
              for t in pan:
                if validate_pan(t):
                  memory['pan'] = str(pan)
                  break
                print("Sorry, your response was not valid.")
            else:
              print('Length of Pan card must be 10')
                
    new_application = pd.DataFrame(index=[0], columns=user_data.columns)
    new_application['mobile'] = memory['mobile']
    new_application['aadhaar'] = memory['aadhaar']
    new_application['name'] = memory['name']
    new_application['pan'] = memory['pan']
    new_application['pin_code'] = memory['pincode']
    user_data = pd.concat([user_data, new_application], ignore_index=True)
    user_data.to_excel('user_data.xlsx')
    print('We have received your application, Our agent will call you back shortly')

#### To output the receipt related response after a valid Customer ID is provided by a user.

In [36]:
def receipt():
    get_id()
    if memory['id']:
      print('Checking payment')
      print('You will receive your payment receipt within 48 hours.')

#### To output the payment reflection related response after a valid Customer ID is provided by a user.

In [37]:
def payment_reflection():
    get_id()
    if memory['id']:    
      print('Checking payment')
      print('Your payment will be reflected in your account within 48 hours.')

#### To output the balance deduction related response after a valid Customer ID is provided by a user.

In [38]:
def twice_deduction():
    get_id() 
    if memory['id']:
      print('Checking payment')
      print('You will receive your balance amount within 7 working days.')

#### To output the electricity complaint related response after a valid Customer ID is provided by a user.

In [39]:
def power_cut():
    get_id()
    if memory['id']:
      #generate ticket 
      print('Your electricity related complaint has been registered. Our agent will reach out to you shortly.')

#### To output the meter complaint related response after a valid Customer ID is provided by a user.

In [40]:
def faulty_meter():
    get_id()
    if memory['id']:    
      print('Your meter related complaint has been registered. Our agent will reach out to you shortly.')

#### To output the bill complaint related response after a valid Customer ID is provided by a user.

In [41]:
def high_bill():
    get_id()
    if memory['id']:
      print('Your bill related complaint has been registered. Our agent will reach out to you shortly.')

### Sentiment check

In [42]:
def sent_polarity(text):
  sent = spacy.load('en_core_web_sm')
  sent.add_pipe('spacytextblob')
  if sent(text)._.subjectivity >0.4:
    return sent(text)._.polarity
  else:
    return 0

## Running the Chatbot

In [57]:
memory = dict.fromkeys(['name', 'id', 'mobile', 'pincode', 'aadhaar', 'pan' ])
n=0
m = 10
while n<=m:
    n+=1
    message = input("YOU: ")
    if message == 'clear':
        clear_memory()
    elif message == 'quit':
        n = m+1
        print('You have ended the conversation')    
    
    if (sent_polarity(message) <= 0.2 and sent_polarity(message) >= -0.2) and message != 'clear' and message != 'quit':
      intent = predict(message)
      predict_proba(message)
      #print(intent)
      if intent == 'greet':
        #common_entity_extraction(message)
        greet()
      elif intent == 'monthlyBill':
        bill_due()
      elif intent == 'momGrowth':  
        mom()
      elif intent == 'consumptionPattern':
        consumption()
      elif intent == 'paymentReceipt':
        receipt()
      elif intent == 'paymentReflection':
        payment_reflection()
      elif intent == 'twiceDeduction':
        twice_deduction()
      elif intent == 'powerCut':
        power_cut()
      elif intent == 'newConnection':
        common_entity_extraction(message)
        new_connection()
      elif intent == 'meterReading':
        reading()
      elif intent == 'faultyMeter':
        faulty_meter()
      elif intent == 'highBill':
        high_bill()
      elif intent == 'fallback':
        print('Sorry, I didn\'t get that. Please try again')

    elif sent_polarity(message) > 0.2 and sent_polarity(message) < 0.6:
        print("Thank you for your kind words.")
    elif sent_polarity(message) >= 0.6:
        print("It's great that you are satisfied with our service.")
    elif sent_polarity(
        
        
        message) < -0.2 and sent_polarity(message) > -0.6:
        print("Sorry to hear that. We are continuously trying to improve ourselves.")
    elif sent_polarity(message) <= -0.6:
        print("We are sorry for your bad experience. Our agent will get in touch with you shortly.")
        print("Also, you can reach out to us on 1800-100-1234 helpline no.")

YOU: Hi
Hi, I am an interactive Chatbot. 
I am here to help you with any questions you may have about utlity, account creation, payment & refund and complaint registration!
What can I help you with today?
YOU: amount payable
Please enter your ID: (To go back type back) 67891
Checking your balance due
Balance due for ID: 67891 is: Rs.10000
YOU: metre readng
Checking your meter reading
Meter reading for ID: 67891 is: 678919
YOU: new connectio
Please enter your name: (To go back type quit) my name is Aatish
Please enter your mobile number: (To go back type quit) 8989012312
Please enter your pincode: (To go back type quit) 234617
Please enter your aadhaar number: (To go back type quit) 789656431243
Please enter your pan number: (To go back type quit) ABC1234F
Length of Pan card must be 10
Please enter your pan number: (To go back type quit) 1234567890
Sorry, your response was not valid.
Please enter your pan number: (To go back type quit) ABCDE1234F
We have received your application, Our a

#### To check data stored in memory